# Job Interview Guide — Targeted ML Workshop
**Senay · ML Specialist interview · 23 October 2026**

## Learning goals and workflow
Your score was **9/15 (60%)**. This notebook targets the six missed questions, rather than repeating topics you answered correctly. A single question is a limited diagnostic, not a complete measure of your ability.

Inspired by your workshops: explanation → small example → reusable function → student TODO → sanity check → interpretation. Everything uses Python's standard library. No credentials, external datasets, custom modules, or package installations are required. Choose a Python 3 kernel in Jupyter or VS Code and run from the top.

Worked examples run immediately. Replace each `None` TODO to activate its checks. A message saying “TODO pending” means the exercise is unfinished, not passed. Reference answers are at the end; attempt the exercises first.

## Interview record
| Q | Topic | Your answer | Correct | Result |
|---|---|---|---|---|
| 1 | Learning paradigms | C | C | Correct |
| 2 | Features and target | B | B | Correct |
| 3 | Train/validation/test | D | D | Correct |
| 4 | Preprocessing leakage | C | C | Correct |
| 5 | Regression interpretation | D | A | Review |
| 6 | MSE | E | E | Correct |
| 7 | R² | D | B | Review |
| 8 | Probability versus threshold | B | C | Review |
| 9 | Cross-entropy | B | D | Review |
| 10 | KNN neighbor count | B | B | Correct |
| 11 | KNN scaling and latency | A | E | Review |
| 12 | Tree leaf predictions | C | C | Correct |
| 13 | Tree complexity | D | D | Correct |
| 14 | Imbalanced evaluation | D | B | Review |
| 15 | Professional judgment | E | E | Correct |


In [1]:
import math
from statistics import mean, pstdev

def close(actual, expected):
    """Check an exercise's numerical result."""
    assert math.isclose(actual, expected, rel_tol=1e-9, abs_tol=1e-9), (actual, expected)

print("Ready: six targeted exercises, no additional packages required.")

Ready: six targeted exercises, no additional packages required.


## 1. Regression predicts a value; it does not guarantee the actual outcome
**Q5: you chose D; correct answer A.**

For $\hat y = 30000 - 2000x$, the intercept is $30,000 and the slope is **−$2,000 per year**. The model predicts a $2,000 decrease for one additional year. It does not prove that every real car loses exactly that amount: condition, mileage, brand, and random variation may also matter.

For a four-year-old car, the predicted price is $22,000. If its actual price is $24,000, the residual, defined here as $y-\hat y$, is +$2,000: the model underpredicted.

A fitted association is not automatically a causal effect. Also, extrapolation far beyond the observed age range can produce unrealistic prices.

In [2]:
def predict_car_price(age):
    return 30_000 - 2_000 * age

prediction = predict_car_price(4)
residual = 24_000 - prediction
close(prediction, 22_000)
close(residual, 2_000)
print(f"Prediction: ${prediction:,.0f}; residual: ${residual:,.0f}")

Prediction: $22,000; residual: $2,000


### Exercise 1 — Price and residual
A six-year-old car sells for $19,500. Calculate its predicted price and residual. Then explain why the residual does not invalidate the meaning of the slope.

In [3]:
# TODO: call predict_car_price and calculate actual minus predicted.
exercise_1_prediction = None
exercise_1_residual = None
if exercise_1_prediction is None or exercise_1_residual is None:
    print("Exercise 1: TODO pending")
else:
    close(exercise_1_prediction, 18_000)
    close(exercise_1_residual, 1_500)
    print("Exercise 1 checks passed; explain the remaining prediction error.")

Exercise 1: TODO pending


## 2. R² compares squared error with a mean baseline
**Q7: you chose D; correct answer B.**

For nonconstant actual targets:
$$R^2=1-\frac{\sum_i(y_i-\hat y_i)^2}{\sum_i(y_i-\bar y)^2}.$$
The numerator is the model's sum of squared errors (SSE). The denominator is the squared error from predicting the **mean of the evaluation set's actual targets** for every row.

- R² = 1: perfect predictions on this set.
- R² = 0: equal squared error to that mean baseline.
- R² < 0: worse squared error than that baseline.

R² = −0.20 means model SSE is **1.20 times** baseline SSE. It does not mean 20% of variation is explained or that every prediction has negative correlation. A deployment baseline must be learned from training data; the evaluation-set mean here defines the R² denominator, not an available production prediction rule.

In [4]:
def r_squared(actual, predicted):
    """Manual R²; reject constant-target examples for clarity."""
    assert len(actual) == len(predicted) and len(actual) > 0
    sse = sum((a - p)**2 for a, p in zip(actual, predicted))
    baseline_sse = sum((a - mean(actual))**2 for a in actual)
    if baseline_sse == 0:
        raise ValueError("This teaching formula requires nonconstant targets.")
    return 1 - sse / baseline_sse

actual = [10, 20, 30]
predicted = [30, 20, 10]
close(r_squared(actual, predicted), -3)
close(r_squared(actual, actual), 1)
close(r_squared(actual, [20, 20, 20]), 0)
print("Reversed predictions R²:", r_squared(actual, predicted))

Reversed predictions R²: -3.0


### Exercise 2 — Interpret negative R²
Baseline SSE is 1,000; model SSE is 1,200. Compute R² and the ratio of model error to baseline error. Write one sentence explaining whether the model beats this baseline.

In [5]:
# TODO: use the R² formula and the ratio model_SSE / baseline_SSE.
exercise_2_r2 = None
exercise_2_ratio = None
if exercise_2_r2 is None or exercise_2_ratio is None:
    print("Exercise 2: TODO pending")
else:
    close(exercise_2_r2, -0.2)
    close(exercise_2_ratio, 1.2)
    print("Exercise 2 checks passed; interpret the 20% increase in squared error.")

Exercise 2: TODO pending


## 3. A classification threshold changes the decision, not the probability
**Q8: you chose B; correct answer C.**

The model outputs P(churn) = 0.70. With the rule $\hat y=1$ when $p\ge t$, a threshold of 0.50 gives churn; 0.80 gives no churn. The underlying probability remains **0.70**. P(no churn) = 0.30 is its complement, not the updated churn probability.

Threshold selection is a decision-policy choice. Raising the threshold yields fewer or equal positive predictions and cannot increase recall on a fixed labeled set. Precision often increases, but is not guaranteed to increase at every step. Select thresholds on validation data, not by repeatedly examining final test results.

In [6]:
def classify(probabilities, threshold=0.5):
    return [int(p >= threshold) for p in probabilities]

probabilities = [0.20, 0.55, 0.70, 0.90]
print("Probabilities:", probabilities)
print("Threshold 0.50:", classify(probabilities, 0.50))
print("Threshold 0.80:", classify(probabilities, 0.80))
assert probabilities == [0.20, 0.55, 0.70, 0.90]

Probabilities: [0.2, 0.55, 0.7, 0.9]
Threshold 0.50: [0, 1, 1, 1]
Threshold 0.80: [0, 0, 0, 1]


### Exercise 3 — Apply two thresholds
For churn probabilities `[0.4, 0.7, 0.8]`, predict labels at thresholds 0.5 and 0.8. Remember the rule includes equality. Does either operation retrain the model?

In [7]:
# TODO: call classify twice.
exercise_3_low = None
exercise_3_high = None
if exercise_3_low is None or exercise_3_high is None:
    print("Exercise 3: TODO pending")
else:
    assert exercise_3_low == [0, 1, 1]
    assert exercise_3_high == [0, 0, 1]
    print("Exercise 3 checks passed; the probabilities remain unchanged.")

Exercise 3: TODO pending


## 4. Cross-entropy penalizes confidence in the wrong outcome
**Q9: you chose B; correct answer D.**

Binary cross-entropy is $-[y\ln(p)+(1-y)\ln(1-p)]$, with $p=P(y=1)$.

For an actual positive ($y=1$), loss is $-\ln(p)$: p=0.90 gives about **0.105**, while p=0.10 gives about **2.303**. The second prediction confidently favors the wrong class and gets the larger penalty.

For an actual negative ($y=0$), loss is $-\ln(1-p)$, so the preference reverses. Log-loss uses probabilities directly; thresholding first discards useful confidence information. Mean log-loss is not a percentage error rate. Clipping is a numerical safeguard at probabilities 0 and 1.

In [8]:
def binary_cross_entropy(label, probability):
    if label not in (0, 1) or not math.isfinite(probability) or not 0 <= probability <= 1:
        raise ValueError("Use a binary label and a finite probability between 0 and 1.")
    p = min(max(probability, 1e-15), 1 - 1e-15)
    return -(label * math.log(p) + (1 - label) * math.log(1 - p))

for p in [0.90, 0.50, 0.10]:
    print(f"Actual=1, P(positive)={p:.2f}, loss={binary_cross_entropy(1,p):.3f}")
assert binary_cross_entropy(1, 0.9) < binary_cross_entropy(1, 0.1)
close(binary_cross_entropy(1, 0.9), binary_cross_entropy(0, 0.1))

Actual=1, P(positive)=0.90, loss=0.105
Actual=1, P(positive)=0.50, loss=0.693
Actual=1, P(positive)=0.10, loss=2.303


### Exercise 4 — Actual negative
A customer did **not** churn. Model A predicts P(churn)=0.9 and Model B predicts P(churn)=0.1. Compute both losses and the mean of those two losses. Explain which probability is more consistent with the observed outcome.

In [9]:
# TODO: call binary_cross_entropy with label=0, then average.
exercise_4_a = None
exercise_4_b = None
exercise_4_mean = None
if any(v is None for v in [exercise_4_a, exercise_4_b, exercise_4_mean]):
    print("Exercise 4: TODO pending")
else:
    close(exercise_4_a, -math.log(0.1))
    close(exercise_4_b, -math.log(0.9))
    close(exercise_4_mean, (-math.log(0.1)-math.log(0.9))/2)
    assert exercise_4_a > exercise_4_b
    print("Exercise 4 checks passed.")

Exercise 4: TODO pending


## 5. KNN depends on distance units and prediction workload
**Q11: you chose A; correct answer E.**

Euclidean distance squares and adds feature differences. A $10,000 income difference can dominate a 20-year age difference simply because dollars have larger numerical units. Being numerical does not make features comparable.

Standardization uses $z=(x-\mu_{train})/\sigma_{train}$. Fit these statistics on training data only; apply the same statistics to validation, test, and incoming observations. Do not scale each customer independently.

Scaling does not guarantee that KNN is the right model. Benchmark accuracy, memory, and latency using representative data and workloads. Brute-force KNN compares a query with training points; other search algorithms can behave differently. Dataset size, dimension, search method, and hardware all affect speed. Logistic regression is a useful alternative to benchmark, not an automatic winner.

In [10]:
# Each row is [annual income, age]; labels are illustrative, not real outcomes.
training = [[30_000, 20], [60_000, 60], [90_000, 40]]
labels = ["A", "B", "C"]
customer = [58_000, 22]

def distance(a, b):
    return math.sqrt(sum((x-y)**2 for x,y in zip(a,b)))

def fit_standardizer(rows):
    columns = list(zip(*rows))
    return [mean(c) for c in columns], [pstdev(c) or 1.0 for c in columns]

def transform(row, means, scales):
    return [(x-m)/s for x,m,s in zip(row, means, scales)]

means, scales = fit_standardizer(training)
scaled_training = [transform(row, means, scales) for row in training]
scaled_customer = transform(customer, means, scales)
raw_distances = [distance(row, customer) for row in training]
scaled_distances = [distance(row, scaled_customer) for row in scaled_training]
raw_nearest = min(range(len(training)), key=lambda i: raw_distances[i])
scaled_nearest = min(range(len(training)), key=lambda i: scaled_distances[i])
print("Raw distances:", [round(v, 2) for v in raw_distances])
print("Scaled distances:", [round(v, 2) for v in scaled_distances])
print("Nearest before scaling:", labels[raw_nearest])
print("Nearest after scaling:", labels[scaled_nearest])
assert labels[raw_nearest] == "B"
assert labels[scaled_nearest] == "A"
for column in zip(*scaled_training):
    close(mean(column), 0)
    close(pstdev(column), 1)

Raw distances: [28000.0, 2000.36, 32000.01]
Scaled distances: [1.15, 2.33, 1.71]
Nearest before scaling: B
Nearest after scaling: A


### Exercise 5 — Transform a new customer
Transform `[60000, 40]` using the existing training statistics. Then write a short benchmark plan: compare KNN and logistic regression using the same valid split, suitable metrics, and measured prediction latency. State why scaling cannot guarantee a latency improvement.

In [11]:
# TODO: reuse means and scales; do not refit on the new customer.
exercise_5_scaled = None
if exercise_5_scaled is None:
    print("Exercise 5: TODO pending")
else:
    assert len(exercise_5_scaled) == 2
    for value in exercise_5_scaled:
        close(value, 0)
    print("Exercise 5 checks passed; add your benchmark plan in Markdown.")

Exercise 5: TODO pending


## 6. Imbalanced evaluation must retain the real evaluation problem
**Q14: you chose D; correct answer B.**

With 1% fraud, always predicting non-fraud gives 99% accuracy but **zero fraud recall**. Removing non-fraud cases from the test set hides false positives and changes the population you are evaluating. It cannot measure realistic precision or false-alarm workload.

Use a representative untouched test set and select decisions on validation data. Inspect the confusion matrix, fraud precision, recall, and error costs:
- Precision = TP / (TP + FP): of flagged transactions, how many are fraud?
- Recall = TP / (TP + FN): of actual frauds, how many were found?

In 10,000 transactions with 100 frauds, suppose a candidate catches 80 frauds and flags 200 legitimate transactions. TP=80, FN=20, FP=200, TN=9700. Its accuracy is 97.8%—lower than the useless majority baseline—but its recall is 80%. Whether it is useful depends on false-alarm costs and review capacity, not accuracy alone.

In [12]:
def classification_metrics(tp, fp, fn, tn):
    total = tp + fp + fn + tn
    assert total > 0 and min(tp,fp,fn,tn) >= 0
    return {
        "accuracy": (tp+tn)/total,
        "precision": tp/(tp+fp) if tp+fp else 0.0,
        "recall": tp/(tp+fn) if tp+fn else 0.0,
    }

baseline = classification_metrics(0, 0, 100, 9900)
candidate = classification_metrics(80, 200, 20, 9700)
print("Always non-fraud:", baseline)
print("Candidate:", candidate)
close(baseline["accuracy"], 0.99)
close(baseline["recall"], 0)
close(candidate["precision"], 80/280)
close(candidate["recall"], 0.8)
close(candidate["accuracy"], 0.978)
# Precision is undefined when no positives are predicted; our teaching function reports 0.

Always non-fraud: {'accuracy': 0.99, 'precision': 0.0, 'recall': 0.0}
Candidate: {'accuracy': 0.978, 'precision': 0.2857142857142857, 'recall': 0.8}


### Exercise 6 — Compare a stricter validation threshold
On the **same validation transactions**, a stricter threshold gives TP=60, FP=40, FN=40, TN=9860. Compute metrics and the number of alerts. Compare with the candidate above: which catches more fraud, and which generates fewer false alarms?

If investigators can review at most 150 alerts per comparable period, discuss the trade-off. Do not claim that one threshold is universally best, or that these validation results guarantee test results.

In [13]:
# TODO: call classification_metrics and calculate TP + FP.
exercise_6_metrics = None
exercise_6_alerts = None
if exercise_6_metrics is None or exercise_6_alerts is None:
    print("Exercise 6: TODO pending")
else:
    close(exercise_6_metrics["precision"], 0.6)
    close(exercise_6_metrics["recall"], 0.6)
    close(exercise_6_metrics["accuracy"], 0.992)
    assert exercise_6_alerts == 100
    print("Exercise 6 checks passed; explain the missed-fraud versus workload trade-off.")

Exercise 6: TODO pending


## Reflection — explain it like an interview answer
Write 4–6 sentences:
1. Which misconception did you correct, and what numerical example convinced you?
2. How do predicted values, probabilities, and final decisions differ?
3. Why can high accuracy or strong training performance be misleading?
4. What checks will you perform before recommending a model?

**My reflection:** _Replace this with your own explanation._

## Reference answers — attempt the TODOs first
1. `predict_car_price(6)` → 18,000; `19500 - exercise_1_prediction` → 1,500. Actual outcomes may differ from the fitted prediction.
2. `1 - 1200/1000` → −0.2; `1200/1000` → 1.2. Model squared error is 20% larger than baseline squared error.
3. `classify([0.4,0.7,0.8],0.5)` → `[0,1,1]`; at 0.8 → `[0,0,1]`. No retraining or probability change is required.
4. `binary_cross_entropy(0,0.9)` → approximately 2.302585; `binary_cross_entropy(0,0.1)` → 0.105361; their mean → 1.203973. Model B assigns higher probability to the observed negative outcome.
5. `transform([60000,40],means,scales)` → `[0.0,0.0]`. Both values equal their training means. Fit preprocessing within training folds when using cross-validation; benchmark inference with the full pipeline included.
6. `classification_metrics(60,40,40,9860)` → accuracy 0.992, precision 0.6, recall 0.6; `60+40` → 100 alerts. The earlier candidate catches 80 rather than 60 frauds, but generates 280 rather than 100 alerts. The stricter threshold fits the stated review limit, while missing more fraud. Confirm business costs and validate on a separate test set after selecting a policy.

## Next practice
Re-explain the six topics without looking at the reference answers. Then try new values in each example. Your successful topics—splitting, leakage, neighbor-count selection, tree predictions, tree complexity, and professional judgment—provide a foundation for this review.